# 前処理実験ノート（ベース複製・時系列CV）

- **元**: `train_baseline.ipynb` を複製。ベースと「加えた前処理」が精度向上に寄与するかを最短で試す実験場。
- **既に試した実験との関係**: `train_extended.ipynb` の PREPROCESS_EXPERIMENTS（movie_info→長さ・語数のみ、movie_title→頻度のみ、〇〇+頻度追加）は**既に実施済み**（結果は docs/01_PREPROCESS.md に記録）。追加実験の**主軸**はそれらの**繰り返しではなく**、**時系列TE・LOO・ビン・欠損2値・TEビン化**など、まだ試していない処理。
- **検証**: 時系列CV（2013〜2016 を val）。**前処理の統計量（TE・頻度・LOO等）は各 fold の学習部分（tr）のみで計算**し、val/test には tr の統計を適用（リーク防止・時系列考慮）。
- **試す前処理**（§11 でアルゴリズム詳細を記載）:
  - **時系列TE**: 過去のみで集計した Target Encoding を **追加**（category は残す）。スムージングあり。
  - **LOO**: critic_name に Leave-One-Out を **追加**。
  - **movie_info → 長さ・語数のみ** / **movie_title → 頻度のみ**: 既存実験と同一。参照用に含める。
  - **ビニング**: `runtime_bin`, `movie_age_bin`, `release_decade`（2値 or ビン）。
  - **欠損2値**: `is_runtime_missing`, `is_movie_age_days_missing`。
  - **TEをビン/2値で**: 時系列TEの値をビン化して単一軸依存を弱める。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from preprocess import load_train_test
from feature_engineering import create_features

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("Setup complete.")

Setup complete.


In [2]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

train = create_features(train)
test = create_features(test)

# ベースと同じ: 8列は category、runtime は中央値で欠損補完
for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype("category")
        test[col] = test[col].fillna("missing").astype("category")
for col in ["runtime"]:
    if col in train.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

print("特徴量作成・前処理完了.")

Train: 653,507, Test: 40,716
特徴量作成・前処理完了.


In [3]:
# ベースの特徴量リスト（train_baseline と同じ）
FEATURES_BASE = [
    "rotten_tomatoes_link", "critic_name", "top_critic", "publisher_name",
    "movie_title", "movie_info", "content_rating", "directors", "authors", "actors",
    "runtime", "production_company",
    "review_year", "review_month", "review_dayofweek",
    "release_year", "release_month", "release_dayofweek", "movie_age_days",
    "genre_Drama", "genre_Comedy", "genre_Action", "genre_Mystery",
    "genre_Fantasy", "genre_Romance", "genre_Horror", "genre_Documentary",
]
print(f"ベース特徴量: {len(FEATURES_BASE)}個")

ベース特徴量: 27個


In [4]:
# 時系列CV: 検証年ごとに「その年より前で学習 → その年で検証」
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [5]:
# 前処理用ヘルパー（時系列・fold の tr のみで統計を計算し val/te に適用）

def _ts_te_col(df_tr, df_val, df_te, col, target_name="target", m=10):
    """時系列TE: tr 内は「その行より前」のみで平均、val/te は tr 全体のカテゴリ別スムージング平均。"""
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    val_arr = df_val[col].astype(str).map(map_).fillna(global_mean).values if len(df_val) else np.array([])
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, val_arr, te_arr

def _loo_col(df_tr, df_val, df_te, col, target_name="target", m=5):
    """LOO: tr は自分除く同カテゴリ平均、val/te は tr のカテゴリ別スムージング平均。"""
    global_mean = float(df_tr[target_name].mean())
    g = df_tr.groupby(col)[target_name]
    s_sum, s_cnt = g.transform("sum"), g.transform("count")
    y = df_tr[target_name].values
    loo = np.where(s_cnt > 1, (s_sum - y) / (s_cnt - 1), global_mean)
    ser_tr = pd.Series((s_cnt * loo + m * global_mean) / (s_cnt + m), index=df_tr.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    val_arr = df_val[col].astype(str).map(map_).fillna(global_mean).values if len(df_val) else np.array([])
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, val_arr, te_arr

def _freq_col(df_tr, df_val, df_te, col):
    """頻度: tr の value_counts で val/te をマッピング。未知は 0。"""
    vc = df_tr[col].astype(str).value_counts()
    ser_tr = df_tr[col].astype(str).map(vc).fillna(0).astype(int)
    val_arr = df_val[col].astype(str).map(vc).fillna(0).astype(int).values if len(df_val) else np.array([])
    te_arr = df_te[col].astype(str).map(vc).fillna(0).astype(int).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, val_arr, te_arr

In [6]:
def build_X_for_experiment(config_name, train_df, test_df, tr_idx, val_idx, y_train):
    """実験設定に応じて X_tr, X_val, X_te と特徴量リストを組み立て。統計は tr のみで計算。
    config_name は str または list/tuple（複数設定の組み合わせ可）。"""
    active = set(config_name) if isinstance(config_name, (list, tuple)) else {config_name}
    if tr_idx is not None:
        tr_df = train_df.iloc[tr_idx].copy()
        val_df = train_df.iloc[val_idx].copy() if val_idx is not None and len(val_idx) else pd.DataFrame()
    else:
        tr_df = train_df.copy()
        val_df = pd.DataFrame()
    te_df = test_df.copy() if test_df is not None else None
    if "target" not in tr_df.columns and y_train is not None:
        tr_df["target"] = y_train[tr_idx] if tr_idx is not None else y_train

    base = [c for c in FEATURES_BASE if c in tr_df.columns]
    extra_feats = []

    if active == {"base"}:
        feats = base
        X_tr = tr_df[feats].copy()
        X_val = val_df[feats].copy() if len(val_df) else None
        X_te = te_df[feats].copy() if te_df is not None else None
        return X_tr, X_val, X_te, feats

    drop_mi = bool(active & {"movie_info_meta", "all_meta_freq"})
    drop_mt = bool(active & {"movie_title_freq", "all_meta_freq"})
    feats = [c for c in base if not (c == "movie_info" and drop_mi) and not (c == "movie_title" and drop_mt)]

    if drop_mi:
        tr_df["movie_info_len"] = tr_df["movie_info"].astype(str).str.len()
        tr_df["movie_info_word_count"] = tr_df["movie_info"].astype(str).str.split().str.len().fillna(0).astype(int)
        if len(val_df): val_df["movie_info_len"] = val_df["movie_info"].astype(str).str.len(); val_df["movie_info_word_count"] = val_df["movie_info"].astype(str).str.split().str.len().fillna(0).astype(int)
        if te_df is not None: te_df["movie_info_len"] = te_df["movie_info"].astype(str).str.len(); te_df["movie_info_word_count"] = te_df["movie_info"].astype(str).str.split().str.len().fillna(0).astype(int)
        extra_feats += ["movie_info_len", "movie_info_word_count"]

    if drop_mt:
        st, va, ta = _freq_col(tr_df, val_df, te_df, "movie_title")
        tr_df["movie_title_freq"] = st
        if len(val_df): val_df["movie_title_freq"] = va
        if te_df is not None: te_df["movie_title_freq"] = ta
        extra_feats.append("movie_title_freq")

    add_bins = bool(active & {"bins", "bins_and_flags", "ts_te_bins", "all_meta_freq"})
    if add_bins:
        tr_df["runtime_bin"] = pd.cut(tr_df["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
        if len(val_df): val_df["runtime_bin"] = pd.cut(val_df["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
        if te_df is not None: te_df["runtime_bin"] = pd.cut(te_df["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
        def ab(x): return 0 if pd.isna(x) or x<0 else 1 if x<365 else 2 if x<365*5 else 3 if x<365*20 else 4
        tr_df["movie_age_bin"] = tr_df["movie_age_days"].apply(ab)
        if len(val_df): val_df["movie_age_bin"] = val_df["movie_age_days"].apply(ab)
        if te_df is not None: te_df["movie_age_bin"] = te_df["movie_age_days"].apply(ab)
        tr_df["release_decade"] = (tr_df["release_year"] // 10 * 10).fillna(1990)
        if len(val_df): val_df["release_decade"] = (val_df["release_year"] // 10 * 10).fillna(1990)
        if te_df is not None: te_df["release_decade"] = (te_df["release_year"] // 10 * 10).fillna(1990)
        extra_feats += ["runtime_bin", "movie_age_bin", "release_decade"]

    add_flags = bool(active & {"missing_flags", "bins_and_flags"})
    if add_flags:
        tr_df["is_runtime_missing"] = 0
        tr_df["is_movie_age_days_missing"] = (tr_df["movie_age_days"].isna()).astype(int)
        if len(val_df): val_df["is_runtime_missing"] = 0; val_df["is_movie_age_days_missing"] = (val_df["movie_age_days"].isna()).astype(int)
        if te_df is not None: te_df["is_runtime_missing"] = 0; te_df["is_movie_age_days_missing"] = (te_df["movie_age_days"].isna()).astype(int)
        extra_feats += ["is_runtime_missing", "is_movie_age_days_missing"]

    if active & {"ts_te", "ts_te_bins", "ts_te_binned"}:
        for col in ["critic_name", "production_company"]:
            if col not in tr_df.columns: continue
            st, va, ta = _ts_te_col(tr_df, val_df, te_df, col, "target", m=10)
            tr_df["{}_te_ts".format(col)] = st.reindex(tr_df.index).fillna(tr_df["target"].mean())
            if len(val_df): val_df["{}_te_ts".format(col)] = va
            if te_df is not None: te_df["{}_te_ts".format(col)] = ta
            extra_feats.append("{}_te_ts".format(col))
        if "ts_te_binned" in active:
            for c in ["critic_name_te_ts", "production_company_te_ts"]:
                if c not in tr_df.columns: continue
                tr_df[c+"_bin"] = pd.cut(tr_df[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
                if len(val_df): val_df[c+"_bin"] = pd.cut(val_df[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
                if te_df is not None: te_df[c+"_bin"] = pd.cut(te_df[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
                extra_feats.append(c+"_bin")

    if "loo" in active:
        st, va, ta = _loo_col(tr_df, val_df, te_df, "critic_name", "target", m=5)
        tr_df["critic_name_loo"] = st
        if len(val_df): val_df["critic_name_loo"] = va
        if te_df is not None: te_df["critic_name_loo"] = ta
        extra_feats.append("critic_name_loo")

    # train_extended の add_freq: category は残し、頻度を追加（add_freq_列名）
    ADD_FREQ_COLS = ["critic_name", "directors", "authors", "actors", "rotten_tomatoes_link", "production_company"]
    for col in ADD_FREQ_COLS:
        if f"add_freq_{col}" not in active or col not in tr_df.columns:
            continue
        st, va, ta = _freq_col(tr_df, val_df, te_df, col)
        tr_df[f"{col}_freq"] = st
        if len(val_df): val_df[f"{col}_freq"] = va
        if te_df is not None: te_df[f"{col}_freq"] = ta
        extra_feats.append(f"{col}_freq")

    feats = feats + [f for f in extra_feats if f in tr_df.columns]
    X_tr = tr_df[feats].copy()
    X_val = val_df[feats].copy() if len(val_df) else None
    X_te = te_df[feats].copy() if te_df is not None else None
    return X_tr, X_val, X_te, feats

In [10]:
# 実験設定一覧（docs §10・§11 / train_extended の PREPROCESS_EXPERIMENTS 相当）
# add_freq は add_freq_列名（critic_name, directors, authors, actors, rotten_tomatoes_link, production_company）
EXPERIMENT_CONFIGS = [
    "base", "ts_te", "ts_te_binned", "ts_te_bins", "loo",
    "movie_info_meta", "movie_title_freq",
    "add_freq_critic_name", "add_freq_directors", "add_freq_authors",
    "add_freq_actors", "add_freq_rotten_tomatoes_link", "add_freq_production_company",
    "bins", "missing_flags", "bins_and_flags", "all_meta_freq",
]
# 提出用「3つ」の設定（3提出 + 3C2 + 3C3 で使用）
THREE_CONFIGS = ["ts_te_binned", "ts_te", "ts_te_bins"]

In [11]:
y = train["target"].values
lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

results = []
for config_name in EXPERIMENT_CONFIGS:
    fold_scores = []
    n_feat = None
    for fold, (tr_idx, val_idx) in enumerate(time_splits):
        X_tr, X_val, _, feats = build_X_for_experiment(config_name, train, test, tr_idx, val_idx, y)
        n_feat = len(feats)
        if X_val is None or len(X_val) == 0:
            continue
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y[tr_idx], eval_set=[(X_val, y[val_idx])], callbacks=[lgb.early_stopping(20, verbose=False)])
        pred = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y[val_idx], pred))
    cv_auc = np.mean(fold_scores) if fold_scores else np.nan
    results.append({"設定": config_name, "n_feat": n_feat, "CV_AUC": round(cv_auc, 4)})
    print(f"  {config_name}: n_feat={n_feat}, CV AUC={cv_auc:.4f}")

df_results = pd.DataFrame(results)
base_auc = df_results.loc[df_results["設定"] == "base", "CV_AUC"].iloc[0]
df_results["差分(ベース比)"] = (df_results["CV_AUC"] - base_auc).round(4)
display(df_results)

  base: n_feat=27, CV AUC=0.7369
  ts_te: n_feat=29, CV AUC=0.7425
  ts_te_binned: n_feat=31, CV AUC=0.7426
  ts_te_bins: n_feat=32, CV AUC=0.7425
  loo: n_feat=28, CV AUC=0.7203
  movie_info_meta: n_feat=28, CV AUC=0.7365
  movie_title_freq: n_feat=27, CV AUC=0.7326
  bins: n_feat=30, CV AUC=0.7369
  missing_flags: n_feat=29, CV AUC=0.7369
  bins_and_flags: n_feat=32, CV AUC=0.7369
  all_meta_freq: n_feat=31, CV AUC=0.7350


,設定,n_feat,CV_AUC,差分(ベース比)
0,base,27,0.7369,0.0000
1,ts_te,29,0.7425,0.0056
2,ts_te_binned,31,0.7426,0.0057
3,ts_te_bins,32,0.7425,0.0056
4,loo,28,0.7203,-0.0166
5,movie_info_meta,28,0.7365,-0.0004
6,movie_title_freq,27,0.7326,-0.0043
7,bins,30,0.7369,0.0000
8,missing_flags,29,0.7369,0.0000
9,bins_and_flags,32,0.7369,0.0000


In [13]:
# ベースより伸びた設定のみ表示
improved = df_results[df_results["差分(ベース比)"] > 0].sort_values("差分(ベース比)", ascending=False)
if len(improved) > 0:
    print("ベースよりCVが伸びた前処理:")
    display(improved)
else:
    print("ベースより伸びた設定はありません。")

ベースよりCVが伸びた前処理:


,設定,n_feat,CV_AUC,差分(ベース比)
2,ts_te_binned,31,0.7426,0.0057
1,ts_te,29,0.7425,0.0056
3,ts_te_bins,32,0.7425,0.0056


In [17]:
# 提出用: 3設定をそれぞれ1本 + 3C2 + 3C3 で学習して保存（THREE_CONFIGS を変えれば一括提出）
import itertools

for cfg in THREE_CONFIGS:
    X_train, _, X_test, feats = build_X_for_experiment(cfg, train, test, None, None, y)
    model_full = lgb.LGBMClassifier(**lgb_params)
    model_full.fit(X_train, y)
    pred = model_full.predict_proba(X_test)[:, 1]
    fname = f"submission_{cfg}.csv"
    pd.DataFrame({"ID": test["ID"], "target": pred}).to_csv(fname, index=False)
    print(f"Saved {fname}")

for (a, b) in itertools.combinations(THREE_CONFIGS, 2):
    cfg = [a, b]
    X_train, _, X_test, feats = build_X_for_experiment(cfg, train, test, None, None, y)
    model_full = lgb.LGBMClassifier(**lgb_params)
    model_full.fit(X_train, y)
    pred = model_full.predict_proba(X_test)[:, 1]
    fname = f"submission_{a}_{b}.csv"
    pd.DataFrame({"ID": test["ID"], "target": pred}).to_csv(fname, index=False)
    print(f"Saved {fname} (3C2)")

X_train, _, X_test, feats = build_X_for_experiment(THREE_CONFIGS, train, test, None, None, y)
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X_train, y)
pred = model_full.predict_proba(X_test)[:, 1]
pd.DataFrame({"ID": test["ID"], "target": pred}).to_csv("submission_3C3.csv", index=False)
print("Saved submission_3C3.csv (3C3)")
print("Done: 3本 + 3C2(3本) + 3C3(1本) = 7提出ファイル")

NameError: name 'THREE_CONFIGS' is not defined